# Tutorial 2 — Agent Enhancement: Tool Use, Multi-Turn, Opik

Tutorial 1 left you at `backend.complete(system, user)`. That is enough for one-shot prompting, but agents add three things on top:

1. **Tool use** — the LLM picks among `@function_tool` definitions and SEOCHO actually runs them. Eight tools ship in `seocho/tools.py` (5 indexing, 2–3 query).
2. **Multi-turn design** — `Session` accumulates entities and relationships across `add()` calls so later `ask()` calls have context. State is structured (entity cache + query cache), not raw chat history.
3. **Observability with Opik** — every `Session.add` / `ask` / `run` logs a span; Opik visualizes them; toggle on/off via env var.

Three execution modes climb that ladder:

| Mode | Entry point | What runs |
|------|-------------|-----------|
| `pipeline` (default) | `Session.add` / `Session.ask` | Deterministic chunk → extract → validate → link → write. No LLM reasoning about flow. |
| `agent` | `Session.add` / `Session.ask` | LLM agent calls the tools. Falls back to pipeline on error. |
| `supervisor` | `Session.run` | Router agent hands off to IndexingAgent or QueryAgent. |

Source references — `seocho/agent_config.py`, `seocho/session.py`, `seocho/agent/factory.py`, `seocho/tools.py`, `seocho/tracing.py`.

## 0. Prerequisites

In [ ]:
# --- Setup (run me first) ---
import sys
from pathlib import Path

try:
    import seocho
    from dotenv import load_dotenv
except ImportError:
    %pip install -q "seocho[local]" python-dotenv
    import seocho
    from dotenv import load_dotenv

load_dotenv(Path.cwd().parent / ".env", override=False)
print('seocho', seocho.__version__, '| .env loaded')


In [6]:
import os, getpass

PROVIDERS = [
    ('openai',   'OPENAI_API_KEY',     'gpt-4o-mini'),
    ('grok',     'XAI_API_KEY',        'grok-4.20-reasoning'),
    ('kimi',     'MOONSHOT_API_KEY',   'kimi-k2.5'),
    ('deepseek', 'DEEPSEEK_API_KEY',   'deepseek-chat'),
]

# Prompt for any keys not already set (.env / shell / earlier cells).
# Press Enter to skip a provider you don't have a key for.
for _, env_name, _ in PROVIDERS:
    if not os.environ.get(env_name):
        val = getpass.getpass(f'{env_name} (blank to skip): ')
        if val:
            os.environ[env_name] = val

AVAILABLE = [(p, env, m) for p, env, m in PROVIDERS if os.environ.get(env)]
LIVE = bool(AVAILABLE)

if LIVE:
    PROVIDER, _, MODEL = AVAILABLE[0]
    LLM_SPEC = f'{PROVIDER}/{MODEL}'
    print(f'live cells will run via {LLM_SPEC}')
else:
    PROVIDER = MODEL = LLM_SPEC = None
    print('[NOTE] no provider key set — live cells will skip.')

live cells will run via openai/gpt-4o-mini


## 1. A small ontology — placeholder for the agent shell

`Session` requires an ontology to construct the indexing/query agents (the tools need to know what entity types exist). For this tutorial we use a hand-rolled Person/Company schema; Tutorial 3 swaps in FIBO and shows the ontology *driving* extraction quality.

In [7]:
from seocho import Ontology, NodeDef, RelDef, P

ontology = Ontology(
    name='finance_demo',
    nodes={
        'Person':  NodeDef(properties={'name': P(str, unique=True), 'title': P(str)}),
        'Company': NodeDef(properties={'name': P(str, unique=True), 'sector': P(str)}),
    },
    relationships={
        'WORKS_AT': RelDef(source='Person', target='Company', cardinality='MANY_TO_ONE'),
    },
)
print('validate :', ontology.validate() or 'OK')

validate : OK


## 2. Pillar 1 — Tool use

Pipeline mode runs a deterministic pipeline; agent mode lets the LLM **decide** when to call which tool. The tools are real Python functions decorated with `@function_tool`. The LLM sees their signatures and docstrings, picks the right ones, and SEOCHO executes them.

**Indexing tools** (`create_indexing_tools()` in `seocho/tools.py`):

1. `extract_entities(text, category)` — LLM extraction against the ontology.
2. `validate_extraction(extraction_json)` — SHACL validation.
3. `score_extraction(extraction_json)` — quality score.
4. `link_entities(extraction_json)` — dedup by `label::name`.
5. `write_to_graph(extraction_json, database, source_id)` — persist.

**Query tools** (`create_query_tools()`):

6. `text2cypher(intent, anchor_entity, ...)` — intent-driven Cypher build.
7. `execute_cypher(cypher, params_json, database)` — read against the graph.
8. `search_similar(query, limit)` — optional, only with a vector store wired.

In [ ]:
from seocho import Seocho, AGENT_PRESETS

TEXT = 'Tim Cook is the CEO of Apple, a technology company.'

if LIVE:
    # Pipeline mode — no tool calls, deterministic pipeline
    s_pipeline = Seocho.local(ontology, llm=LLM_SPEC, user_id='openai-yitae')
    with s_pipeline.session('pipeline-demo') as sess:
        out = sess.add(TEXT)
        print('[pipeline ]', f"mode={out.get('mode')}  nodes={out.get('nodes_created')}  rels={out.get('relationships_created')}")
    s_pipeline.close()

    # Agent mode — LLM picks among the 5 indexing tools
    s_agent = Seocho.local(ontology, llm=LLM_SPEC, user_id='openai-yitae', agent_config=AGENT_PRESETS['agent'])
    with s_agent.session('agent-demo') as sess:
        out = sess.add(TEXT)
        print('[agent    ]', f"mode={out.get('mode')}  nodes={out.get('nodes_created')}  rels={out.get('relationships_created')}  degraded={out.get('degraded', False)}")
    s_agent.close()
else:
    print('[SKIP] tool-use demo — set OPENAI_API_KEY to run.')

If the agent path raises (tool failure, network blip, Agents SDK not installed), `Session._add_via_agent` catches it, calls `_add_via_pipeline` instead, and stamps `degraded=True`, `fallback_from='agent'`, `fallback_reason=<exc>` on the result. Read these fields to know whether your run actually used the agent.

## 3. Pillar 2 — Multi-turn design

A `Session` is not a chat history. It is a structured cache of (a) extracted entities by source, (b) extracted relationships, and (c) recent answers. When you call `session.ask(...)`, the prior `add()` calls have already populated this cache, and the agent receives it as context — without you re-shipping any text.

Three things to watch:

- The **entity cache** grows with every `add()`. The agent can answer about subjects from earlier turns.
- The **query cache** memoizes recent answers — repeating the same `ask()` returns instantly.
- The **session id and database** are propagated to every span so traces (next pillar) are grouped automatically.

In [ ]:
if LIVE:
    s = Seocho.local(ontology, llm=LLM_SPEC, user_id='openai-yitae')
    with s.session('multi-turn-demo') as sess:
        # Turn 1 — establish facts
        sess.add('Tim Cook is the CEO of Apple, a technology company.')
        # Turn 2 — extend
        sess.add('Jensen Huang leads NVIDIA, an AI hardware company.')
        # Turn 3 — refer back
        sess.add('Apple sells iPhones; NVIDIA sells GPUs.')

        # Inspect the cache the next ask() will see
        ctx = sess.context.to_agent_context(ontology=ontology)
        print('--- structured context the agent receives at ask() ---')
        print(ctx[:400] + ('...' if len(ctx) > 400 else ''))
        print()

        # Multi-turn answer — references content from turn 1 + 2
        print('answer (cross-turn):', sess.ask('Compare the CEOs of Apple and NVIDIA.'))

        # Cache hit — repeating the question returns instantly
        print('answer (cache hit):', sess.ask('Compare the CEOs of Apple and NVIDIA.'))
    s.close()
else:
    print('[SKIP] multi-turn demo — set OPENAI_API_KEY to run.')

Inspecting `sess.context.history` and `sess.context.entity_index` after a session reveals what got cached. The relevant fields:

- `add_indexing(source_id, nodes, rels, content, mode, degraded, ...)` — recorded after each `add()`.
- `add_query(question, answer, mode, degraded, ...)` — recorded after each `ask()` (cache adds use `mode='cache'`).
- `register_entities(extracted_nodes, source_id, db)` — populates the entity cache the agent reads on the next turn.

## 4. Pillar 3 — Observability with Opik

Up to now everything is invisible — you see one answer per call, but not the path it took. SEOCHO ships a vendor-neutral tracing layer (`seocho/tracing.py`) with four backends — `none`, `console`, `jsonl`, `opik` — and you can run several at once.

The pattern:

1. **Off by default.** No tracing until you call `enable_tracing(...)` or set `SEOCHO_TRACE_BACKEND`.
2. **JSONL is the canonical artifact.** Portable, grep-able, commit-friendly under `traces/`.
3. **Opik is the optional team exporter.** Set `OPIK_API_KEY` (or configure `~/.opik.config`) and add `"opik"` to the backend list.
4. **`disable_tracing()`** turns everything off again.

Every `Session.add` / `ask` / `run` call logs a span carrying the active `agent_config` (including `routing_policy` weights), so policy effects become visible in the trace stream.

In [ ]:
from seocho.tracing import (
    enable_tracing, disable_tracing,
    is_tracing_enabled, is_backend_enabled, current_backend_names,
    flush_tracing,
)

# On/off mode, in priority order:
#   TUTORIAL_TRACE='off'   → no tracing
#   TUTORIAL_TRACE='opik'  → Opik + JSONL (requires OPIK_API_KEY)
#   TUTORIAL_TRACE='jsonl' → JSONL only
#   anything else (auto)   → Opik+JSONL if OPIK_API_KEY is set, else JSONL
TUTORIAL_TRACE = os.environ.get('TUTORIAL_TRACE', 'auto').lower()
JSONL_PATH = './traces/tutorial_02.jsonl'

if TUTORIAL_TRACE == 'off':
    disable_tracing()
elif TUTORIAL_TRACE == 'opik' or (TUTORIAL_TRACE == 'auto' and os.environ.get('OPIK_API_KEY')):
    enable_tracing(
        backend=['opik', 'jsonl'],
        output=JSONL_PATH,
        project_name='seocho-openai',
    )
else:
    enable_tracing(backend='jsonl', output=JSONL_PATH)

print('tracing enabled  :', is_tracing_enabled())
print('active backends  :', current_backend_names() or '(none)')
print('opik backend on? :', is_backend_enabled('opik'))
print('JSONL artifact   :', JSONL_PATH)

## 5. RoutingPolicy — a trace-side concern

`RoutingPolicy(latency, token_efficiency, information_quality)` weights are read by the supervisor's system prompt to decide retries, validation strictness, and answer style. The effect is invisible from a single answer; it shows up in traces as differences in span count, retry frequency, and elapsed time.

Three named presets:

- `RoutingPolicy.fast()` — `latency=0.7`.
- `RoutingPolicy.balanced()` — equal weights.
- `RoutingPolicy.thorough()` — `information_quality=0.8`, retries enabled.

In [ ]:
from seocho import RoutingPolicy, AgentConfig

for label, policy in [
    ('fast    ', RoutingPolicy.fast()),
    ('balanced', RoutingPolicy.balanced()),
    ('thorough', RoutingPolicy.thorough()),
]:
    h = policy.to_agent_hints()
    print(f'{label}  dominant={policy.dominant_axis:<22}  '
          f'qual_threshold={h["extraction_quality_threshold"]:.2f}  '
          f'retries={h["extraction_max_retries"]}  '
          f'repair={h["repair_budget"]}  '
          f'validation={h["validation_on_fail"]:<6}  '
          f'answer={h["answer_style"]}')

Run the supervisor under two contrasting policies — the same call, different policy, distinct trace shapes. With Opik enabled you see two traces in the project dashboard; with JSONL only, `tail -f traces/tutorial_02.jsonl | jq` shows the spans live.

In [ ]:
if LIVE:
    for label, policy in [
        ('fast    ', RoutingPolicy.fast()),
        ('thorough', RoutingPolicy.thorough()),
    ]:
        cfg = AgentConfig(execution_mode='supervisor', handoff=True, routing_policy=policy)
        s = Seocho.local(ontology, llm=LLM_SPEC, user_id='openai-yitae', agent_config=cfg)
        with s.session(f'trace-{label.strip()}') as sess:
            sess.run('Index this fact: Sundar Pichai is the CEO of Google.')
            answer = sess.run('Who is the CEO of Google?')
        s.close()
        print(f'  [{label}] policy={policy.dominant_axis:<22} answer_preview={answer[:80]!r}')
    flush_tracing()
    print()
    if is_backend_enabled('opik'):
        print('Spans sent to Opik project ' + repr('seocho-openai') + ' and JSONL at ' + JSONL_PATH)
    else:
        print('Spans written to JSONL at ' + JSONL_PATH + ' (set OPIK_API_KEY to also send to Opik).')
else:
    print('[SKIP] tracing demo — set OPENAI_API_KEY to run.')

**Turn it off when you are done.** Tracing has overhead; Opik writes go over the network. Leaving it enabled in interactive sessions is a foot-gun.

In [ ]:
disable_tracing()
print('tracing enabled  :', is_tracing_enabled())
print('active backends  :', current_backend_names() or '(none)')

## 6. When to pick which mode

| You want… | Pick |
|-----------|------|
| Deterministic batch ingest, lowest cost, no retries | `pipeline` (default) |
| Single-task adaptivity (re-extract on low score, repair empty queries) | `agent` |
| Conversational front-door that decides whether to index or query | `supervisor` |
| Strict latency SLO | `supervisor_fast` or `agent` + `RoutingPolicy.fast()` |
| Highest-quality extraction, willing to spend tokens | `strict` or `supervisor_thorough` |

All three modes route through `extraction.agents_runtime.get_agents_runtime().run(...)`, so trace records, observability hooks, and degraded-state reporting work identically.

**Next:** Tutorial 3 brings in a real ontology (FIBO). Same agent, same multi-turn design, same Opik traces — but the system prompt and the tools' SHACL validation are now driven by a versioned schema with a `context_hash` you can check at every span.